# Análisis de coherencia de outliers (CRIM, ZN, B)

La regla de IQR sola marca demasiados "outliers" en variables muy asimétricas (CRIM, ZN, B) que en realidad son la cola larga de una distribución real, no errores. Este análisis agrega un segundo filtro: para cada outlier, chequea si su relación con el target (MEDV) y con otras variables correlacionadas por dominio va en la dirección que el propio EDA (matriz de correlación) predice.

**Corrección importante**: la fracción de relaciones rotas se calcula sobre las relaciones que efectivamente se pudieron *evaluar* (ignorando NaN), no sobre un conteo absoluto. Así, una fila con una sola relación evaluable que la rompe pesa tanto como una con tres relaciones evaluables que rompe las tres — antes, una fila con datos faltantes en las variables secundarias quedaba injustamente protegida de ser marcada como incoherente.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Datos

Igual que en el resto del pipeline: nunca se imputa MEDV, se dropean sus nulos primero, antes de analizar cualquier otra cosa.

In [ ]:
df = pd.read_csv('house-prices-tp.csv')
df_limpio = df.dropna(subset=['MEDV']).copy()

## Función: detectar outliers por IQR

Marca los outliers de una columna con la regla clásica (Q1 - 1.5·IQR, Q3 + 1.5·IQR) y guarda de qué lado cae cada uno ('alto' o 'bajo'), que se usa después para juzgar coherencia.

In [ ]:
def detectar_outliers_iqr(df, columna):
    serie = df[columna].dropna()
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df_outliers = df[(df[columna] < lim_inf) | (df[columna] > lim_sup)].copy()
    df_outliers['lado'] = np.where(df_outliers[columna] > lim_sup, 'alto', 'bajo')
    return df_outliers, lim_inf, lim_sup

## Función: chequear coherencia

Para los outliers de una columna, evalúa si el valor extremo es coherente con la relación esperada respecto al target y a otras variables correlacionadas por dominio (según la matriz de correlación del EDA).

- `direccion`: +1 si la columna correlaciona POSITIVO con el target, -1 si correlaciona NEGATIVO (esto sale de la matriz de correlación, no es un supuesto arbitrario).
- `relaciones`: dict opcional `{nombre_var: signo_esperado}` con otras variables relacionadas por dominio (ej: `{'LSTAT': 1, 'RM': -1}` para CRIM).
- `umbral_fraccion`: proporción mínima de relaciones **evaluables** que se deben romper para marcar la fila como incoherente (0.5 = rompe la mayoría, 1.0 = rompe absolutamente todo lo que se pudo chequear).

Cada outlier se compara contra la **mediana global** de cada variable (no la mediana de los outliers). Las filas con NaN en la variable comparada no cuentan ni a favor ni en contra: simplemente reducen el denominador de la fracción, en vez de bajar el conteo absoluto como en una primera versión de este análisis (que subestimaba la incoherencia de filas con más datos faltantes).

In [ ]:
def chequear_coherencia(df, columna, direccion, relaciones=None, target='MEDV',
                         umbral_fraccion=0.5):
    df_outliers, lim_inf, lim_sup = detectar_outliers_iqr(df, columna)
    signo_lado = np.where(df_outliers['lado'] == 'alto', 1, -1)

    def marca_rompe(var, signo_esperado):
        mediana_var = df[var].median(skipna=True)
        diff = df_outliers[var] - mediana_var
        signo_fila = np.sign(diff)
        signo_esperado_fila = signo_lado * signo_esperado
        evaluable = diff.notna() & (signo_fila != 0)
        rompe = evaluable & (np.sign(signo_esperado_fila) != signo_fila)
        return evaluable, rompe

    todas_las_relaciones = [(target, direccion)] + list((relaciones or {}).items())
    n_evaluables = pd.Series(0, index=df_outliers.index)
    n_rotas = pd.Series(0, index=df_outliers.index)
    for var, signo_var in todas_las_relaciones:
        evaluable, rompe = marca_rompe(var, signo_var)
        df_outliers[f'rompe_{var}'] = rompe
        n_evaluables += evaluable.astype(int)
        n_rotas += rompe.astype(int)

    df_outliers['n_evaluables'] = n_evaluables
    df_outliers['n_rotas'] = n_rotas
    df_outliers['fraccion_rota'] = np.where(n_evaluables > 0, n_rotas / n_evaluables, 0.0)

    df_incoherentes = (
        df_outliers[(df_outliers['n_evaluables'] > 0) &
                    (df_outliers['fraccion_rota'] >= umbral_fraccion)]
        .sort_values(['fraccion_rota', 'n_evaluables'], ascending=False)
    )

    print(f"--- {columna} ---")
    print(f"  límites IQR: [{lim_inf:.2f}, {lim_sup:.2f}]  -> {len(df_outliers)} outliers totales")
    print(f"  incoherentes (rompen >={umbral_fraccion*100:.0f}% de lo evaluable): {len(df_incoherentes)}")

    return df_outliers, df_incoherentes

## Función: graficar coherencia

Extiende el scatterplot "Impacto de X en el valor de la propiedad" ya existente, distinguiendo tres grupos: datos normales, outliers coherentes (se conservan) y outliers incoherentes (candidatos a revisión manual).

In [ ]:
def graficar_coherencia(df, columna, df_outliers, df_incoherentes, target='MEDV'):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=columna, y=target, color='lightgrey', alpha=0.6,
                     label='Datos normales')

    coherentes = df_outliers.drop(df_incoherentes.index)
    sns.scatterplot(data=coherentes, x=columna, y=target, color='orange', edgecolor='black',
                     label='Outlier coherente (conservar)')

    sns.scatterplot(data=df_incoherentes, x=columna, y=target, color='red', edgecolor='black',
                     s=90, label='Outlier incoherente (revisar)')

    plt.title(f'Coherencia de outliers de {columna} respecto a {target}')
    plt.xlabel(columna)
    plt.ylabel(target)
    plt.legend()
    plt.show()

## Configuración: signos esperados por variable

Estos signos salen de la propia matriz de correlación del EDA (Sección "¿Cuál es la correlación entre las variables?"), no son un supuesto arbitrario:

- **CRIM** correlaciona negativo con MEDV, negativo con RM, positivo con LSTAT.
- **ZN** correlaciona positivo con MEDV, negativo con INDUS, negativo con LSTAT.
- **B** correlaciona positivo con MEDV, negativo con LSTAT.

In [ ]:
config = {
    'CRIM': dict(direccion=-1, relaciones={'RM': -1, 'LSTAT': 1}),
    'ZN':   dict(direccion=+1, relaciones={'INDUS': -1, 'LSTAT': -1}),
    'B':    dict(direccion=+1, relaciones={'LSTAT': -1}),
}

## Ejecutar el análisis para CRIM, ZN y B

`umbral_fraccion=0.5`: marca como incoherente a la fila que rompe la **mayoría** de las relaciones que se pudieron evaluar (no todas). Subilo a 1.0 si preferís ser más estricto y quedarte solo con las filas que rompen absolutamente todo lo que se pudo chequear — con ese umbral la lista de candidatas es más chica, y puede que la triangulación entre variables dé vacía.

In [ ]:
resultados = {}
for col, cfg in config.items():
    outliers, incoherentes = chequear_coherencia(df_limpio, col, **cfg, umbral_fraccion=0.5)
    resultados[col] = (outliers, incoherentes)
    graficar_coherencia(df_limpio, col, outliers, incoherentes)

## Triangulación entre variables

Una fila que rompe el patrón esperado en **más de una** variable independiente es un candidato mucho más fuerte a revisión que una fila que solo lo rompe en una. No es automático que se elimine: es el punto de partida para decidir con criterio (diagnóstico de influencia, regresión robusta, o exclusión justificada y comparando métricas con/sin esas filas).

In [ ]:
indices = [set(incoh.index) for _, incoh in resultados.values()]
interseccion = set.union(*[
    indices[i] & indices[j]
    for i in range(len(indices)) for j in range(i + 1, len(indices))
])

print("--- Triangulación: filas incoherentes en MÁS DE UNA variable ---")
print(f"Filas candidatas a revisión prioritaria: {sorted(interseccion)}")

cols_mostrar = ['CRIM', 'ZN', 'B', 'RM', 'LSTAT', 'INDUS', 'MEDV']
df_limpio.loc[sorted(interseccion), cols_mostrar]